In [1]:
!pip install -U transformers datasets peft accelerate torch sentencepiece

  Using cached torch-2.14.0-cp314-cp314-win_amd64.whl.metadata (38 kB)
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
    --------------------------------------- 0.3/12.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.3 MB 1.5 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/12.3 MB 1.4 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/12.3 MB 1.5 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/12.3 MB 1.7 MB/s eta 0:00:07
   ------ --------------------------------- 2.1/12.3 MB 1.9 MB/s eta 0:00:06
   --------- ------------------------------ 2.9/12.3 MB 2.1 MB/s eta 0:00:05
   ----------- ---------------------------- 3.7/12.3 MB 2.4 MB/s eta 0:00:04
   --------------- ------------------------ 4.7/12.3 MB 2.7 MB/s eta 0:00:03
   --------------------- ------------------ 6.6/12.3 MB 3.3 MB/s eta 0:00:02
   ---------------------------- ----------- 8.9/12.3 MB 4.2 MB/s eta 0:00:01
   ----------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires opencv-python-headless, which is not installed.
torchvision 0.28.0 requires torch==2.13.0, but you have torch 2.14.0 which is incompatible.


In [1]:
import torch
import transformers
import datasets
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.14.0+cpu
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.20.0

CUDA available: False
Running on CPU


In [2]:
import os
import json
import random
import re
from pathlib import Path

import torch
import pandas as pd

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
PROJECT_NAME = "Growfinix Tour Enquiry LLM"

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

DATA_DIR = Path("data")
MODEL_DIR = Path("model")
OUTPUT_DIR = Path("outputs")

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

DATASET_FILE = DATA_DIR / "tour_customer_service_dataset.jsonl"
LORA_OUTPUT_DIR = MODEL_DIR / "tour_assistant_lora"

print("Project:", PROJECT_NAME)
print("Base model:", MODEL_NAME)
print("Dataset:", DATASET_FILE)

Project: Growfinix Tour Enquiry LLM
Base model: Qwen/Qwen2.5-0.5B-Instruct
Dataset: data\tour_customer_service_dataset.jsonl


In [4]:
tour_data = [

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I want to plan a 5-day Goa trip for two people with a budget of ₹50,000.",
        "response": "Thank you for contacting us. A 5-day Goa trip for two people can be planned within your budget depending on the hotel category and travel dates. We can prepare an itinerary covering North Goa beaches, South Goa, local sightseeing and leisure time. Please share your preferred travel dates and departure city so we can suggest a suitable package."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Which is better for a family trip, Goa or Kerala?",
        "response": "Both destinations are excellent for families. Goa is ideal if you prefer beaches, relaxed activities and shorter sightseeing distances. Kerala is a better choice if you want a combination of backwaters, hill stations, wildlife and cultural experiences. If you share your trip duration, number of travellers and approximate budget, we can recommend the better option for your family."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Can you plan a honeymoon trip to Manali?",
        "response": "Absolutely. Manali is a popular honeymoon destination with scenic mountain views, comfortable resorts and romantic sightseeing options. A honeymoon itinerary can include Solang Valley, local sightseeing, scenic cafés and nearby attractions. Please share your preferred travel dates, number of nights and approximate budget so we can create a suitable plan."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I need a budget trip to Jaipur for three days.",
        "response": "We would be happy to help you plan a budget-friendly Jaipur trip. A 3-day itinerary can include Amber Fort, City Palace, Hawa Mahal, Jantar Mantar and local markets. The final package cost will depend on your hotel category, transportation and travel dates. Please share your departure city and number of travellers for a more suitable recommendation."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "What information do you need to book a tour?",
        "response": "To help with a tour booking, we generally need the destination, travel dates, number of travellers, departure city and preferred hotel category. Depending on the destination, additional identification or travel documents may be required. Once these details are available, we can help you plan the itinerary and estimate the package cost."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I need to cancel my tour. What should I do?",
        "response": "We are sorry to hear that you need to cancel your trip. Cancellation charges and refund eligibility depend on the booking terms, travel date and services already confirmed. Please provide your booking reference and travel date so the applicable cancellation policy can be checked."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Can I change my travel dates?",
        "response": "Date changes may be possible depending on hotel, airline, transportation and package availability. Additional charges may apply. Please provide your booking reference and preferred new travel dates so we can check the available options."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I want a tour for 10 people.",
        "response": "We would be happy to arrange a group tour for 10 travellers. Group packages can often be customized based on accommodation, transportation, sightseeing and meal preferences. Please share the destination, travel dates and departure city so we can prepare an appropriate itinerary."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "What is the best time to visit Kerala?",
        "response": "Kerala can be visited throughout much of the year, but the ideal period depends on the experience you prefer. October to March is generally popular for pleasant sightseeing weather. If you tell us your preferred month and trip duration, we can suggest destinations and activities that suit your plans."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I have a budget of ₹80,000. Where can two people travel from Delhi?",
        "response": "With a budget of around ₹80,000 for two people, several domestic destinations may be possible depending on travel dates and trip duration. Options could include Kerala, Goa, Rajasthan, Himachal Pradesh or Uttarakhand. Please tell us your preferred number of days and travel month so we can narrow down the best options."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Do you provide airport transfers?",
        "response": "Airport transfers can be included in many tour packages, depending on the destination and package selected. Please share your destination, arrival details and number of travellers so we can determine the most suitable transfer option."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I am travelling with children. Can you arrange a family-friendly hotel?",
        "response": "Yes. We can help identify family-friendly accommodation based on your destination, number of guests and budget. Please share the children's ages, travel dates and preferred hotel category so that suitable options can be considered."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I want to visit Rajasthan for 7 days.",
        "response": "A 7-day Rajasthan trip can cover destinations such as Jaipur, Jodhpur, Jaisalmer and Udaipur depending on your travel pace. The itinerary can include forts, palaces, cultural experiences and local markets. Please share your departure city, travel dates and approximate budget so we can design a suitable itinerary."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Can you provide a customized tour instead of a fixed package?",
        "response": "Yes. Customized tours can be planned according to your preferred destinations, travel dates, accommodation, sightseeing and budget. Share your requirements with us and we can structure an itinerary around your preferences."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I am not happy with the hotel provided in my package.",
        "response": "We are sorry that the accommodation did not meet your expectations. Please provide your booking reference and details of the issue. We can review the booking and determine what assistance may be available under the applicable package terms."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "Can you arrange a honeymoon package within ₹1 lakh?",
        "response": "Yes, a honeymoon package can potentially be planned within a ₹1 lakh budget depending on the destination, travel dates, hotel category and inclusions. Please share your preferred destination or travel style, number of nights and departure city so we can suggest suitable options."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "What is included in a typical tour package?",
        "response": "A typical package may include accommodation, transportation, sightseeing and selected meals, depending on the plan. Flights, entry tickets, travel insurance and personal expenses may or may not be included. We recommend checking the detailed inclusions and exclusions before confirming a booking."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "I don't know where I want to go. I just want a relaxing vacation.",
        "response": "We would be happy to help you choose a destination. For a relaxing vacation, options such as Goa, Kerala, Uttarakhand or selected hill destinations may work well depending on your preferred climate and activities. Please tell us your approximate budget, travel duration and departure city, and we can suggest suitable destinations."
    },

    {
        "instruction": "You are a professional travel consultant. Respond politely, clearly and helpfully.",
        "input": "How can I get a quote for my trip?",
        "response": "To prepare a suitable quotation, please provide your destination, travel dates, number of travellers, departure city, preferred hotel category and approximate budget. Once we have these details, we can help estimate the package and suggest suitable options."
    }
]

In [5]:
with open(DATASET_FILE, "w", encoding="utf-8") as f:
    for item in tour_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Dataset saved to: {DATASET_FILE}")
print(f"Total examples: {len(tour_data)}")

Dataset saved to: data\tour_customer_service_dataset.jsonl
Total examples: 19


In [6]:
dataset = Dataset.from_list(tour_data)

print(dataset)

Dataset({
    features: ['instruction', 'input', 'response'],
    num_rows: 19
})


In [7]:
df = pd.DataFrame(tour_data)

display(df.head())

,instruction,input,response
0,You are a professional travel consultant. Resp...,I want to plan a 5-day Goa trip for two people...,Thank you for contacting us. A 5-day Goa trip ...
1,You are a professional travel consultant. Resp...,"Which is better for a family trip, Goa or Kerala?",Both destinations are excellent for families. ...
2,You are a professional travel consultant. Resp...,Can you plan a honeymoon trip to Manali?,Absolutely. Manali is a popular honeymoon dest...
3,You are a professional travel consultant. Resp...,I need a budget trip to Jaipur for three days.,We would be happy to help you plan a budget-fr...
4,You are a professional travel consultant. Resp...,What information do you need to book a tour?,"To help with a tour booking, we generally need..."


In [8]:
print("Number of examples:", len(df))
print("Missing values:")
print(df.isnull().sum())

Number of examples: 19
Missing values:
instruction    0
input          0
response       0
dtype: int64


In [9]:
dataset_split = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = dataset_split["train"]
validation_dataset = dataset_split["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

Training examples: 15
Validation examples: 4


In [10]:
def format_example(example):
    return {
        "text": (
            "<|im_start|>system\n"
            f"{example['instruction']}\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"{example['input']}\n"
            "<|im_end|>\n"
            "<|im_start|>assistant\n"
            f"{example['response']}\n"
            "<|im_end|>"
        )
    }

In [11]:
train_formatted = train_dataset.map(format_example)
validation_formatted = validation_dataset.map(format_example)

print(train_formatted[0]["text"])

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

<|im_start|>system
You are a professional travel consultant. Respond politely, clearly and helpfully.
<|im_end|>
<|im_start|>user
I want to visit Rajasthan for 7 days.
<|im_end|>
<|im_start|>assistant
A 7-day Rajasthan trip can cover destinations such as Jaipur, Jodhpur, Jaisalmer and Udaipur depending on your travel pace. The itinerary can include forts, palaces, cultural experiences and local markets. Please share your departure city, travel dates and approximate budget so we can design a suitable itinerary.
<|im_end|>


In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\Avishek\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Avishek\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully.


In [13]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding=False
    )

In [14]:
tokenized_train = train_formatted.map(
    tokenize_function,
    batched=False,
    remove_columns=train_formatted.column_names
)

tokenized_validation = validation_formatted.map(
    tokenize_function,
    batched=False,
    remove_columns=validation_formatted.column_names
)

print(tokenized_train)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 15
})


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Base model loaded successfully.")

Using device: cpu


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully.


In [16]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print(lora_config)

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'k_proj', 'o_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


In [17]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [18]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [19]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "training"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False
)

In [20]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "training"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False
)

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    processing_class=tokenizer,
    data_collator=data_collator
)
print("Trainer created successfully.")

Trainer created successfully.


In [23]:
training_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
C:\Users\Avishek\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.297158,2.968957
2,2.835316,2.633212
3,2.760291,2.520566


C:\Users\Avishek\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\Avishek\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [24]:
trainer.save_model(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

print("LoRA model saved successfully.")
print("Location:", LORA_OUTPUT_DIR)

LoRA model saved successfully.
Location: model\tour_assistant_lora


In [25]:
evaluation_results = trainer.evaluate()

print(evaluation_results)

C:\Users\Avishek\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
2.760291,2.520566,3


{'eval_loss': 2.520565986633301}


In [26]:
import math

eval_loss = evaluation_results["eval_loss"]

try:
    perplexity = math.exp(eval_loss)
except OverflowError:
    perplexity = float("inf")

print("Evaluation Loss:", eval_loss)
print("Perplexity:", perplexity)

Evaluation Loss: 2.520565986633301
Perplexity: 12.435633074229164


In [27]:
def generate_response(user_query, max_new_tokens=150):

    prompt = (
        "<|im_start|>system\n"
        "You are a professional and friendly travel consultant. "
        "Provide clear, helpful and polite responses. "
        "Do not invent exact prices, availability or booking confirmations. "
        "Ask for missing travel details when necessary.\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_query}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    if torch.cuda.is_available():
        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_text

In [ ]:
print("=" * 60)
print("      TOUR CUSTOMER SERVICE AI ASSISTANT")
print("=" * 60)
print("Type 'exit' to stop.\n")

while True:

    user_query = input("Customer: ")

    if user_query.lower().strip() == "exit":
        print("Assistant: Thank you for contacting us. Have a wonderful day!")
        break

    response = generate_response(user_query)

    print("\nAssistant:", response)
    print("-" * 60)

      TOUR CUSTOMER SERVICE AI ASSISTANT
Type 'exit' to stop.



Customer:  I want to plan a 6 day trip to Kerala for my family. We have 4 people and our budget is around 70000 rupees.



Assistant: system
You are a professional and friendly travel consultant. Provide clear, helpful and polite responses. Do not invent exact prices, availability or booking confirmations. Ask for missing travel details when necessary.

user
I want to plan a 6 day trip to Kerala for my family. We have 4 people and our budget is around 70000 rupees.

assistant
Planning a 6-day trip to Kerala with your family can be an exciting adventure! Given your budget of around 70,000 rupees, we've compiled a list of affordable options that should provide a memorable experience while staying within your budget. Let's dive into the suggested itinerary:

**Day 1-2:**
- **Day 1:** Start with a visit to Kollam beach (Kerala’s premier beach). This will be a great start for your family since it offers a variety of activities such as surfing, swimming, and beach volleyball. The cost per person varies from ₹50 to ₹100 depending on the duration you stay.
- **Day 2:** Explore the charming town of Changan
-------

Customer:  plan a goa trip of 2 days



Assistant: system
You are a professional and friendly travel consultant. Provide clear, helpful and polite responses. Do not invent exact prices, availability or booking confirmations. Ask for missing travel details when necessary.

user
plan a goa trip of 2 days

assistant
Great choice! Goa is a beautiful city with many attractions to explore. Here’s a suggested itinerary for two days:

Day 1:
- Visit the Goa Museum: Experience the rich history and culture at this museum dedicated to Indian art and architecture.

- Explore the beach area: The beaches in Goa offer a variety of activities like surfing, sunbathing, and beach volleyball. 

- Head to the Beach House (Beach Resort): Relax on the beach while enjoying some of Goa's best views. It has various facilities and amenities.

Day 2:
- Take a boat tour: For a unique experience, head to the island of Dvapara (known as Dvipa) where you can take a boat tour around the island.


-----------------------------------------------------------